# Interview Prep: Optimization Questions

This notebook contains 20 common interview questions about optimization in machine learning, plus 3 coding challenges.

**Instructions:**
1. Try to answer each question before revealing the solution
2. Practice explaining concepts out loud
3. Implement the coding challenges without looking at solutions first

---

In [ ]:
# Setup
import numpy as np
import sys
sys.path.insert(0, '..')

from src.optimizers import (
    SGD, Adam, AdamW, RMSprop, Adagrad,
    clip_grad_norm_, compute_grad_norm
)

np.random.seed(42)
print("Setup complete!")

---

## Conceptual Questions (1-20)

---

### Question 1: Explain Adam vs SGD

**Interview Question:** "Can you explain the key differences between Adam and SGD? When would you choose one over the other?"

<details>
<summary><b>Click for Answer</b></summary>

**Key Differences:**

| Aspect | SGD | Adam |
|--------|-----|------|
| Learning rate | Same for all parameters | Adaptive per-parameter |
| Momentum | Optional, single momentum term | Built-in: first moment (momentum) + second moment |
| Memory | O(n) with momentum | O(2n) - stores m and v |
| Convergence | Slower but often better final result | Faster early convergence |
| Hyperparameters | lr, momentum | lr, beta1, beta2, eps |

**When to choose SGD:**
- Training CNNs from scratch (ResNet, VGG)
- When you need best generalization
- Large-scale training with good LR schedule
- When memory is constrained

**When to choose Adam:**
- Starting a new project (works out of box)
- NLP tasks, especially transformers (with AdamW)
- Reinforcement learning
- Sparse gradients
- When you need fast iteration

**Key insight:** Adam often trains faster but may generalize worse. SGD with good tuning often achieves better test accuracy but requires more effort to tune.
</details>

---

### Question 2: What causes exploding gradients?

**Interview Question:** "What causes exploding gradients and how do you fix it?"

<details>
<summary><b>Click for Answer</b></summary>

**Causes of Exploding Gradients:**

1. **RNNs/Deep Networks:** Gradients multiply through layers during backprop. If weight matrices have spectral norm > 1, gradients grow exponentially.

2. **Poor Weight Initialization:** Weights too large cause activations and gradients to explode.

3. **High Learning Rate:** Large updates can push weights into regions with steep gradients.

4. **Unstable Architecture:** Some architectures (deep residual networks without proper init) can have gradient explosion.

**Solutions:**

1. **Gradient Clipping:** Most common solution
   ```python
   clip_grad_norm_(grads, max_norm=1.0)
   ```

2. **Proper Initialization:** Xavier/Glorot for tanh/sigmoid, He for ReLU

3. **Batch Normalization:** Stabilizes activations and gradients

4. **LSTM/GRU instead of vanilla RNN:** Gating mechanisms prevent gradient explosion

5. **Lower Learning Rate:** Reduces update magnitude

6. **Gradient Penalty:** Add regularization term that penalizes large gradients (used in WGANs)

**Detection:** Monitor gradient norms during training. If they suddenly spike to large values or inf/nan, you have exploding gradients.
</details>

---

### Question 3: How do you choose learning rate?

**Interview Question:** "How do you select an appropriate learning rate for your model?"

<details>
<summary><b>Click for Answer</b></summary>

**Strategies for Choosing Learning Rate:**

1. **Start with defaults:**
   - SGD: 0.1 with momentum
   - Adam: 0.001
   - Fine-tuning: 10-100x smaller than training from scratch

2. **Learning Rate Range Test (LR Finder):**
   - Start with very small LR (1e-7)
   - Increase exponentially each batch
   - Plot loss vs LR
   - Choose LR where loss is decreasing fastest (before it starts increasing)

3. **Grid Search / Random Search:**
   - Try powers of 10: [1e-4, 1e-3, 1e-2, 1e-1]
   - Narrow down with finer search

4. **Rule of thumb scaling:**
   - Linear scaling with batch size: `lr = base_lr * (batch_size / 256)`
   - Square root scaling for Adam: `lr = base_lr * sqrt(batch_size / 256)`

5. **Signs of wrong LR:**
   - Too high: Loss explodes, oscillates wildly, or NaN
   - Too low: Training very slow, loss barely decreases

6. **Use LR scheduling:**
   - Warmup: Start low, increase over first few epochs
   - Decay: Step decay, cosine annealing, or reduce on plateau

**Best Practice:** Use a learning rate finder, then use LR scheduling (warmup + cosine decay is a good default).
</details>

---

### Question 4: What is momentum in SGD?

**Interview Question:** "Explain momentum in SGD. Why does it help?"

<details>
<summary><b>Click for Answer</b></summary>

**What is Momentum?**

Momentum adds a "velocity" term to SGD that accumulates gradients from previous steps:

```
v_t = momentum * v_{t-1} + gradient_t
w_{t+1} = w_t - lr * v_t
```

Typical value: 0.9 (averages over ~10 gradient steps)

**Why it helps:**

1. **Accelerates convergence:** In directions with consistent gradients, momentum builds up speed, allowing faster traversal of flat regions.

2. **Dampens oscillations:** In directions where gradients oscillate (like ravines), momentum smooths out the zigzag pattern.

3. **Escapes local minima:** The accumulated velocity can help "roll through" small local minima.

4. **Reduces variance:** Averaging over multiple gradients reduces the noise from mini-batch sampling.

**Physical Analogy:** Imagine a ball rolling down a hilly landscape. Without momentum, it moves purely based on local slope. With momentum, it builds up speed on downhills and can coast through small uphills.

**Nesterov Momentum:** A variant that "looks ahead" before computing the gradient:
```
v_t = momentum * v_{t-1} + gradient(w_t - lr * momentum * v_{t-1})
```
This often gives slightly better convergence.
</details>

---

### Question 5: What is the difference between L2 regularization and weight decay?

**Interview Question:** "Are L2 regularization and weight decay the same thing? Explain."

<details>
<summary><b>Click for Answer</b></summary>

**Short answer:** They're equivalent for SGD, but DIFFERENT for adaptive optimizers like Adam!

**L2 Regularization:**
Add penalty term to loss: `L_total = L + (lambda/2) * ||w||^2`

Gradient becomes: `grad = grad_L + lambda * w`

**Weight Decay:**
Directly shrink weights after each update: `w = w - lr * grad - lr * lambda * w`

**For SGD, they're equivalent:**
```
w = w - lr * (grad + lambda * w)  # L2 reg
w = w - lr * grad - lr * lambda * w  # Weight decay
# Both give: w = w - lr * grad - lr * lambda * w
```

**For Adam, they're different!**

With L2 reg in Adam:
- The weight penalty gets scaled by the adaptive learning rate
- Large gradient parameters get less regularization

With decoupled weight decay (AdamW):
- Weight decay applied uniformly
- Not scaled by adaptive rate
- Provides true regularization effect

**This is why AdamW exists:** It decouples weight decay from the gradient-based update, providing better regularization for Adam.

**Paper:** "Decoupled Weight Decay Regularization" (Loshchilov & Hutter, 2019)
</details>

---

### Question 6: Explain batch normalization's role in optimization

**Interview Question:** "How does batch normalization affect optimization?"

<details>
<summary><b>Click for Answer</b></summary>

**What Batch Normalization Does:**
Normalizes each layer's inputs to have zero mean and unit variance:
```
y = gamma * (x - mean) / sqrt(var + eps) + beta
```

**Optimization Benefits:**

1. **Allows higher learning rates:** Normalized activations are more stable, so larger updates don't cause explosion.

2. **Reduces internal covariate shift:** (Original claim) Layer inputs don't change distribution as much during training. (Note: This explanation is debated.)

3. **Smooths the loss landscape:** Recent research shows BatchNorm makes the loss surface smoother with more predictable gradients.

4. **Reduces sensitivity to initialization:** Since activations are normalized, bad initialization has less impact.

5. **Implicit regularization:** Batch statistics add noise, similar to dropout.

**Gradient Flow Perspective:**
- Without BN: Gradients can vanish/explode through layers
- With BN: Gradients are more uniform across layers

**Trade-offs:**
- Adds computation
- Batch-size dependent (problematic for small batches)
- Behaves differently in training vs inference

**Alternatives:** Layer Norm (better for NLP), Group Norm (batch-size independent), Instance Norm (for style transfer)
</details>

---

### Question 7: What is the vanishing gradient problem?

**Interview Question:** "Explain the vanishing gradient problem and how modern architectures address it."

<details>
<summary><b>Click for Answer</b></summary>

**The Problem:**
In deep networks, gradients can become exponentially small as they propagate backward through layers. This makes early layers train very slowly or not at all.

**Why it happens:**

1. **Activation function saturation:** Sigmoid/tanh have derivatives near 0 for large inputs. Chain rule multiplies these small values: 0.1^10 = 10^-10

2. **Weight matrices:** If ||W|| < 1, repeated multiplication shrinks gradients.

**Solutions:**

1. **ReLU activation:** Derivative is 1 for positive inputs, no saturation. (But can have "dying ReLU" problem)

2. **Residual connections (ResNets):** 
   ```
   y = F(x) + x  # Skip connection
   ```
   Gradients can flow directly through the skip connection.

3. **LSTM/GRU for RNNs:** Gating mechanisms allow gradients to flow unchanged through time.

4. **Batch/Layer Normalization:** Keeps activations in a good range.

5. **Proper initialization:** He init for ReLU, Xavier for tanh/sigmoid.

6. **Gradient clipping:** Doesn't solve vanishing but prevents related instabilities.

**Modern perspective:** Residual connections are the most impactful solution, enabling training of 100+ layer networks.
</details>

---

### Question 8: What are the components of Adam?

**Interview Question:** "Walk me through the Adam update rule step by step."

<details>
<summary><b>Click for Answer</b></summary>

**Adam = Adaptive Moment Estimation**

Combines two ideas:
- Momentum (first moment)
- RMSprop (second moment)

**Step-by-step:**

1. **Update first moment (momentum):**
   ```
   m_t = beta1 * m_{t-1} + (1 - beta1) * g_t
   ```
   Exponential moving average of gradients. Beta1 = 0.9 typically.

2. **Update second moment (squared gradients):**
   ```
   v_t = beta2 * v_{t-1} + (1 - beta2) * g_t^2
   ```
   Exponential moving average of squared gradients. Beta2 = 0.999 typically.

3. **Bias correction:**
   ```
   m_hat = m_t / (1 - beta1^t)
   v_hat = v_t / (1 - beta2^t)
   ```
   Corrects for initialization at zero. Important in early steps.

4. **Update weights:**
   ```
   w = w - lr * m_hat / (sqrt(v_hat) + eps)
   ```
   - Numerator: momentum-corrected gradient direction
   - Denominator: adaptive learning rate scaling

**Intuition:**
- First moment provides momentum (direction)
- Second moment provides per-parameter learning rate scaling
- Parameters with large gradients get smaller effective LR
- Parameters with small gradients get larger effective LR
</details>

---

### Question 9: Why does Adam sometimes not generalize as well as SGD?

**Interview Question:** "I've heard that SGD often generalizes better than Adam. Why might this be?"

<details>
<summary><b>Click for Answer</b></summary>

**Observed phenomenon:** On many image classification benchmarks, well-tuned SGD achieves better test accuracy than Adam, despite Adam training faster.

**Hypotheses:**

1. **Sharp vs flat minima:**
   - Adam can converge to sharper minima (lower loss but poor generalization)
   - SGD's noise helps find flatter minima (better generalization)
   - Sharp minima: small perturbations cause large loss increases
   - Flat minima: more robust to perturbations

2. **Effective learning rate:**
   - Adam's adaptive LR can become very small for frequently-updated parameters
   - This might cause under-training of some important features

3. **Implicit regularization of SGD:**
   - SGD with small batches has inherent noise
   - This noise acts as regularization
   - Adam's moment estimates reduce this noise

4. **Weight decay interaction:**
   - L2 regularization interacts poorly with Adam (solved by AdamW)
   - Standard Adam with weight decay doesn't regularize properly

**Practical implications:**
- Use Adam for fast experimentation
- Use SGD (with momentum and LR schedule) for final training when accuracy matters
- AdamW is a good middle ground, especially for transformers
</details>

---

### Question 10: What is learning rate warmup?

**Interview Question:** "What is learning rate warmup and why is it used?"

<details>
<summary><b>Click for Answer</b></summary>

**What is warmup?**
Starting with a very small learning rate and gradually increasing it to the target value over the first few epochs/steps.

```
if step < warmup_steps:
    lr = base_lr * (step / warmup_steps)
else:
    lr = scheduled_lr(step)
```

**Why use warmup?**

1. **Adam's moment estimates are biased early:**
   - m and v start at zero
   - Bias correction helps but isn't perfect
   - Large LR + poor estimates = bad updates

2. **Network instability at start:**
   - Random initialization means early gradients are noisy
   - Large updates based on random gradients can be harmful

3. **Large batch training:**
   - When scaling to large batches, linear LR scaling can cause instability
   - Warmup allows gradual increase to high LR

4. **Transformers especially need it:**
   - Attention weights can have very large gradients initially
   - Warmup prevents explosion

**Typical settings:**
- 1-10% of total training steps
- Linear warmup is most common
- Can also use exponential or polynomial

**Example:** For 10000 steps total, warmup over first 1000 steps.
</details>

---

### Question 11: What is the role of epsilon in Adam?

**Interview Question:** "What does epsilon do in Adam, and when might you need to change it?"

<details>
<summary><b>Click for Answer</b></summary>

**Role of epsilon:**
```
w = w - lr * m / (sqrt(v) + eps)
```
Epsilon prevents division by zero when v is very small.

**Default value:** 1e-8

**When to change it:**

1. **Mixed precision training (float16):**
   - 1e-8 can underflow in float16
   - Use 1e-4 or larger

2. **Very sparse gradients:**
   - Some parameters may have v very close to 0
   - Larger epsilon provides more stability

3. **Reinforcement learning:**
   - Often use eps=1e-5 for stability
   - Rewards/gradients can be very noisy

4. **When training is unstable:**
   - Try increasing epsilon to 1e-4 or 1e-3
   - Acts as additional regularization on effective LR

**Intuition:**
- Small epsilon: Trust adaptive LR more, can have very large effective LR for low-variance parameters
- Large epsilon: More uniform learning rates, closer to SGD behavior

**Note:** Some frameworks put epsilon inside sqrt: `sqrt(v + eps)` vs `sqrt(v) + eps`. This affects behavior when v is very small.
</details>

---

### Question 12: What is cosine annealing?

**Interview Question:** "Explain cosine annealing for learning rate scheduling."

<details>
<summary><b>Click for Answer</b></summary>

**Cosine Annealing:**
Learning rate follows a cosine curve from initial value to minimum:

```
lr = lr_min + 0.5 * (lr_max - lr_min) * (1 + cos(pi * t / T))
```

Where t = current step, T = total steps.

**Properties:**
- Starts at lr_max
- Smoothly decreases to lr_min
- Slower decay at start and end, faster in middle

**Why it works:**

1. **Early training:** High LR allows fast exploration of loss landscape

2. **Middle training:** Gradual decrease allows settling toward good minima

3. **Late training:** Very low LR allows fine-tuning within the minimum

**Cosine Annealing with Warm Restarts (SGDR):**
- Reset LR back to high value periodically
- Allows escaping local minima
- Can increase period length after each restart

**Comparison to step decay:**
- Step decay: Sudden drops, need to choose when
- Cosine: Smooth, no hyperparameters for when to decay
- Often achieves similar or better results

**Usage:** Very popular for training transformers and CNNs. Combined with warmup: warmup -> cosine decay.
</details>

---

### Question 13: What is gradient accumulation?

**Interview Question:** "Explain gradient accumulation and when you would use it."

<details>
<summary><b>Click for Answer</b></summary>

**Gradient Accumulation:**
Computing gradients over multiple mini-batches before updating weights.

```python
accumulation_steps = 4
optimizer.zero_grad()

for i, batch in enumerate(dataloader):
    loss = model(batch) / accumulation_steps
    loss.backward()  # Gradients accumulate
    
    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()
```

**Effective batch size:** actual_batch * accumulation_steps

**When to use:**

1. **GPU memory limited:**
   - Can't fit large batch in memory
   - Accumulate gradients over smaller batches
   - Same effect as large batch training

2. **Transformer training:**
   - BERT/GPT require large effective batch sizes
   - May need to accumulate over many steps

3. **Consistent training dynamics:**
   - Want to compare with different hardware
   - Match effective batch size for reproducibility

**Important considerations:**
- Divide loss by accumulation steps (or scale LR)
- BatchNorm statistics computed per actual batch, not accumulated batch
- Increases training time proportionally

**Trade-off:** Saves memory, but slower than true large batch (no parallelism across accumulated batches).
</details>

---

### Question 14: Compare RMSprop and Adam

**Interview Question:** "How is Adam related to RMSprop?"

<details>
<summary><b>Click for Answer</b></summary>

**RMSprop:**
```
v = alpha * v + (1 - alpha) * g^2
w = w - lr * g / (sqrt(v) + eps)
```

**Adam:**
```
m = beta1 * m + (1 - beta1) * g      # Added: first moment
v = beta2 * v + (1 - beta2) * g^2    # Same as RMSprop
m_hat = m / (1 - beta1^t)            # Added: bias correction
v_hat = v / (1 - beta2^t)            # Added: bias correction
w = w - lr * m_hat / (sqrt(v_hat) + eps)
```

**Key differences:**

| Aspect | RMSprop | Adam |
|--------|---------|------|
| First moment | No | Yes (momentum) |
| Bias correction | No | Yes |
| Memory | O(n) | O(2n) |
| Convergence | Good | Often better |

**Adam = RMSprop + Momentum + Bias Correction**

**When RMSprop might be preferred:**
- Simpler, fewer hyperparameters
- Slightly less memory
- Sometimes works well for RNNs

**Historical note:** RMSprop was proposed by Hinton in a Coursera lecture (not a paper). Adam was published in 2015 and quickly became the default adaptive optimizer.
</details>

---

### Question 15: What is the saddle point problem?

**Interview Question:** "What are saddle points and why are they problematic for optimization?"

<details>
<summary><b>Click for Answer</b></summary>

**Saddle Points:**
Points where the gradient is zero, but it's not a local minimum or maximum. The surface curves up in some directions and down in others.

**Example in 2D:** z = x^2 - y^2 has a saddle at origin.

**Why they're problematic:**

1. **Gradient is zero:** Gradient descent can slow down dramatically or get stuck.

2. **More common in high dimensions:** In a d-dimensional space, each eigenvalue of the Hessian can be positive or negative. The probability of ALL being positive (local min) decreases exponentially with d.

3. **Flat regions around saddles:** Not just the exact saddle point, but the entire region near it has small gradients.

**Why it's often NOT a big problem in practice:**

1. **SGD noise helps:** Stochastic gradients provide perturbations that can push away from saddles.

2. **Momentum helps:** Accumulated velocity can carry through flat regions.

3. **Saddles aren't "sticky":** Most saddles have directions of negative curvature, so random perturbations eventually escape.

4. **Local minima aren't bad:** In deep learning, most local minima have similar loss values and generalize well.

**Modern view:** Saddle points slow down training but don't prevent convergence. More concerning are "bad" minima with poor generalization.
</details>

---

### Question 16: What is the purpose of bias correction in Adam?

**Interview Question:** "Why does Adam need bias correction?"

<details>
<summary><b>Click for Answer</b></summary>

**The Problem:**

Adam's moment estimates start at zero:
- m_0 = 0
- v_0 = 0

After first step:
- m_1 = (1 - beta1) * g_1 = 0.1 * g_1 (with beta1=0.9)
- This is biased toward zero!

**Expected value analysis:**
```
E[m_t] = E[g] * (1 - beta1^t)
```

At t=1: E[m_1] = 0.1 * E[g] (heavily biased)
At t=10: E[m_10] = 0.65 * E[g] (still biased)
At t=100: E[m_100] = 0.9999... * E[g] (nearly unbiased)

**Bias correction:**
```
m_hat = m_t / (1 - beta1^t)
v_hat = v_t / (1 - beta2^t)
```

This makes E[m_hat] = E[g], removing the bias.

**Why it matters:**

1. **Early training:** Without correction, first few updates use heavily biased estimates, potentially going in wrong direction or with wrong magnitude.

2. **Second moment especially:** beta2=0.999 means v_t is biased for hundreds of steps without correction.

3. **Affects learning rate:** v_hat in denominator affects effective learning rate. Biased v means wrong LR.

**Note:** Some implementations skip bias correction for simplicity. This usually works but can cause issues in early training.
</details>

---

### Question 17: Explain the difference between batch, mini-batch, and stochastic gradient descent

**Interview Question:** "What's the difference between batch GD, mini-batch GD, and SGD?"

<details>
<summary><b>Click for Answer</b></summary>

**Batch Gradient Descent:**
- Uses entire dataset to compute gradient
- One update per epoch
- Gradient is exact (no noise)

**Stochastic Gradient Descent (SGD):**
- Uses single sample to compute gradient
- N updates per epoch (N = dataset size)
- Very noisy gradient

**Mini-batch Gradient Descent:**
- Uses small batch (32-512 samples typically)
- N/B updates per epoch (B = batch size)
- Balance between noise and efficiency

**Comparison:**

| Aspect | Batch | Mini-batch | Stochastic |
|--------|-------|------------|------------|
| Updates/epoch | 1 | N/B | N |
| Gradient noise | None | Medium | High |
| GPU utilization | High | High | Low |
| Memory needed | High | Medium | Low |
| Convergence | Smooth | Slightly noisy | Very noisy |
| Generalization | Worse | Good | Good (noise helps) |

**In practice:** Almost everyone uses mini-batch, but calls it "SGD." Pure batch GD is rarely used (too slow, doesn't generalize as well). Pure stochastic is inefficient on GPUs.

**Why mini-batch works:**
- Gradient estimate is accurate enough
- Some noise helps escape local minima
- Efficient GPU parallelization
</details>

---

### Question 18: What is Adagrad and when would you use it?

**Interview Question:** "Explain Adagrad. What are its advantages and limitations?"

<details>
<summary><b>Click for Answer</b></summary>

**Adagrad Update Rule:**
```
G = G + g^2                    # Accumulate squared gradients
w = w - lr * g / (sqrt(G) + eps)
```

**Key idea:** Scale learning rate inversely with the sum of past squared gradients.

**Advantages:**

1. **Sparse features:** Parameters with rare gradients get larger effective LR. Great for NLP embeddings where some words appear rarely.

2. **Per-parameter LR:** Each parameter gets its own adaptive rate.

3. **No manual LR decay:** LR naturally decreases over time.

4. **Convex optimization:** Proven convergence guarantees for convex problems.

**Limitations:**

1. **Learning rate dies:** G accumulates forever, so LR keeps decreasing. Eventually LR becomes too small to learn.

2. **Not good for deep learning:** The dying LR problem makes it unsuitable for non-convex problems.

3. **Can't "forget" bad gradients:** Early noisy gradients affect the entire training.

**When to use:**
- Training word embeddings
- Sparse feature problems
- Convex optimization
- Problems where training should naturally slow down

**RMSprop/Adam fix the dying LR problem:** Use exponential moving average instead of sum, so old gradients are forgotten.
</details>

---

### Question 19: What is second-order optimization and why isn't it used in deep learning?

**Interview Question:** "Why don't we use second-order methods like Newton's method for deep learning?"

<details>
<summary><b>Click for Answer</b></summary>

**First-order methods (SGD, Adam):**
- Use only gradient (first derivative)
- Update: w = w - lr * gradient

**Second-order methods (Newton's method):**
- Use gradient AND Hessian (second derivative)
- Update: w = w - H^(-1) * gradient
- Hessian contains curvature information

**Why second-order is better theoretically:**
- Accounts for curvature of loss surface
- Converges in fewer steps
- No learning rate needed (in pure form)

**Why it's not used in deep learning:**

1. **Memory:** Hessian is n x n where n = number of parameters. For 100M parameters, that's 10^16 elements!

2. **Computation:** Computing Hessian is O(n^2), inverting it is O(n^3).

3. **Approximations exist but have issues:**
   - L-BFGS: Approximates H^(-1), works for smaller problems
   - K-FAC: Kronecker-factored approximation, promising but complex
   - Natural gradient: Uses Fisher information matrix

4. **SGD works well enough:** With good schedules and momentum, first-order methods work surprisingly well.

5. **Stochasticity helps:** Noisy gradients provide implicit regularization that second-order methods might lose.

**Interesting note:** Adam can be seen as a diagonal approximation to second-order methods, scaling each parameter by its gradient variance.
</details>

---

### Question 20: How do you debug a model that isn't training?

**Interview Question:** "Your model's loss isn't decreasing. Walk me through your debugging process."

<details>
<summary><b>Click for Answer</b></summary>

**Systematic Debugging Checklist:**

**1. Overfit a single batch first:**
```python
single_batch = next(iter(dataloader))
for i in range(1000):
    loss = train_step(single_batch)
# Should reach near-zero loss
```
If this fails, problem is in model/training, not data.

**2. Check gradients:**
- Are gradients None? (computation graph broken)
- Are gradients zero? (dead ReLUs, vanishing gradients)
- Are gradients huge? (exploding gradients)
- Are gradients NaN? (numerical issues)

**3. Check learning rate:**
- Too high: loss oscillates or explodes
- Too low: loss barely moves
- Try LR finder or orders of magnitude

**4. Check data:**
- Labels correct? (common bug: mislabeled data)
- Preprocessing correct? (normalization, scaling)
- Data loading correct? (shuffle, batching)

**5. Check loss function:**
- Right loss for the task?
- Numerical stability? (log(0), division by 0)
- Reduction correct? (mean vs sum)

**6. Check architecture:**
- Activations saturating?
- Skip connections working?
- Dimensions correct?

**7. Check optimizer state:**
- Adam moments initialized?
- Using correct parameter groups?
- Learning rate schedule working?

**Pro tip:** Add extensive logging early. Monitor loss, gradient norms, activation distributions, learning rate every N steps.
</details>

---

## Coding Challenges (1-3)

---

### Coding Challenge 1: Implement SGD with Momentum from Scratch

Implement SGD with momentum without using the provided optimizer classes.

In [ ]:
# YOUR IMPLEMENTATION HERE

def sgd_momentum_step(params, grads, velocities, lr, momentum):
    """
    Perform one step of SGD with momentum.
    
    Args:
        params: List of parameter arrays
        grads: List of gradient arrays
        velocities: List of velocity arrays (same shape as params)
        lr: Learning rate
        momentum: Momentum coefficient
    
    Returns:
        Updated velocities list
    
    Update rules:
        v = momentum * v + grad
        param = param - lr * v
    """
    # TODO: Implement this
    pass

# Test your implementation
def test_sgd_momentum():
    np.random.seed(42)
    W = np.random.randn(10, 5)
    v = np.zeros_like(W)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    losses = []
    for epoch in range(100):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        grad = (2 / len(X)) * X.T @ (y_pred - y)
        
        v = sgd_momentum_step([W], [grad], [v], lr=0.01, momentum=0.9)
        if v is None:
            print("Not implemented yet!")
            return
        v = v[0]
    
    print(f"Initial loss: {losses[0]:.4f}")
    print(f"Final loss: {losses[-1]:.4f}")
    print(f"Loss decreased: {losses[-1] < losses[0]}")

test_sgd_momentum()

<details>
<summary><b>Click for Solution</b></summary>

```python
def sgd_momentum_step(params, grads, velocities, lr, momentum):
    new_velocities = []
    
    for param, grad, v in zip(params, grads, velocities):
        # Update velocity
        v_new = momentum * v + grad
        
        # Update parameter
        param -= lr * v_new
        
        new_velocities.append(v_new)
    
    return new_velocities
```
</details>

In [ ]:
# SOLUTION

def sgd_momentum_step_solution(params, grads, velocities, lr, momentum):
    new_velocities = []
    
    for param, grad, v in zip(params, grads, velocities):
        # Update velocity: v = momentum * v + grad
        v_new = momentum * v + grad
        
        # Update parameter: param = param - lr * v
        param -= lr * v_new
        
        new_velocities.append(v_new)
    
    return new_velocities

# Verify solution
def test_solution():
    np.random.seed(42)
    W = np.random.randn(10, 5)
    v = np.zeros_like(W)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    losses = []
    for epoch in range(100):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        grad = (2 / len(X)) * X.T @ (y_pred - y)
        [v] = sgd_momentum_step_solution([W], [grad], [v], lr=0.01, momentum=0.9)
    
    print(f"Solution - Initial loss: {losses[0]:.4f}")
    print(f"Solution - Final loss: {losses[-1]:.4f}")

test_solution()

---

### Coding Challenge 2: Implement Learning Rate Finder

Implement a learning rate finder that gradually increases LR and records the loss.

In [ ]:
# YOUR IMPLEMENTATION HERE

def lr_finder(model_fn, data, start_lr=1e-7, end_lr=10, num_steps=100):
    """
    Find optimal learning rate by increasing LR exponentially.
    
    Args:
        model_fn: Function that takes (params, X, y) and returns (loss, grads)
        data: Tuple of (X, y)
        start_lr: Starting learning rate
        end_lr: Ending learning rate
        num_steps: Number of steps
    
    Returns:
        lrs: List of learning rates
        losses: List of corresponding losses
    
    Algorithm:
        1. Initialize parameters
        2. For each step:
           - Compute current LR (exponentially increasing)
           - Compute loss and gradients
           - Update parameters with current LR
           - Record LR and loss
    """
    # TODO: Implement this
    pass

# Test your implementation
def test_lr_finder():
    # Simple linear model
    np.random.seed(42)
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    def model_fn(params, X, y):
        W = params[0]
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        grad = (2 / len(X)) * X.T @ (y_pred - y)
        return loss, [grad]
    
    result = lr_finder(model_fn, (X, y))
    if result is None:
        print("Not implemented yet!")
        return
    
    lrs, losses = result
    
    # Find best LR (where loss decreases fastest)
    min_loss_idx = np.argmin(losses)
    print(f"Best LR around: {lrs[min_loss_idx]:.2e}")
    print(f"Lowest loss: {losses[min_loss_idx]:.4f}")

test_lr_finder()

<details>
<summary><b>Click for Solution</b></summary>

```python
def lr_finder(model_fn, data, start_lr=1e-7, end_lr=10, num_steps=100):
    X, y = data
    
    # Initialize parameters
    W = np.random.randn(X.shape[1], y.shape[1]) * 0.01
    params = [W]
    
    # Compute LR multiplier for exponential increase
    lr_mult = (end_lr / start_lr) ** (1 / num_steps)
    
    lrs = []
    losses = []
    lr = start_lr
    
    for step in range(num_steps):
        # Compute loss and gradients
        loss, grads = model_fn(params, X, y)
        
        # Record
        lrs.append(lr)
        losses.append(loss)
        
        # Stop if loss explodes
        if np.isnan(loss) or loss > 100 * losses[0]:
            break
        
        # Update parameters
        for p, g in zip(params, grads):
            p -= lr * g
        
        # Increase LR
        lr *= lr_mult
    
    return lrs, losses
```
</details>

In [ ]:
# SOLUTION

def lr_finder_solution(model_fn, data, start_lr=1e-7, end_lr=10, num_steps=100):
    X, y = data
    
    # Initialize parameters
    W = np.random.randn(X.shape[1], y.shape[1]) * 0.01
    params = [W]
    
    # Compute LR multiplier for exponential increase
    lr_mult = (end_lr / start_lr) ** (1 / num_steps)
    
    lrs = []
    losses = []
    lr = start_lr
    
    for step in range(num_steps):
        # Compute loss and gradients
        loss, grads = model_fn(params, X, y)
        
        # Record
        lrs.append(lr)
        losses.append(loss)
        
        # Stop if loss explodes
        if np.isnan(loss) or loss > 100 * losses[0]:
            break
        
        # Update parameters
        for p, g in zip(params, grads):
            p -= lr * g
        
        # Increase LR
        lr *= lr_mult
    
    return lrs, losses

# Test solution
def test_solution_lr():
    np.random.seed(42)
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    def model_fn(params, X, y):
        W = params[0]
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        grad = (2 / len(X)) * X.T @ (y_pred - y)
        return loss, [grad]
    
    lrs, losses = lr_finder_solution(model_fn, (X, y))
    
    min_loss_idx = np.argmin(losses)
    print(f"Solution - Best LR around: {lrs[min_loss_idx]:.2e}")
    print(f"Solution - Lowest loss: {losses[min_loss_idx]:.4f}")
    print(f"\nLR range tested: {lrs[0]:.2e} to {lrs[-1]:.2e}")
    print(f"Loss range: {min(losses):.4f} to {max(losses):.4f}")

test_solution_lr()

---

### Coding Challenge 3: Implement Adam from Scratch

Implement the Adam optimizer without using the provided optimizer classes.

In [ ]:
# YOUR IMPLEMENTATION HERE

class SimpleAdam:
    """
    Simple Adam optimizer implementation.
    
    Args:
        params: List of parameter arrays
        lr: Learning rate (default: 0.001)
        beta1: First moment decay (default: 0.9)
        beta2: Second moment decay (default: 0.999)
        eps: Numerical stability term (default: 1e-8)
    """
    
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        
        # TODO: Initialize m and v for each parameter
        self.m = None
        self.v = None
        self.t = 0  # Time step
    
    def step(self, grads):
        """
        Perform one optimization step.
        
        Args:
            grads: List of gradients corresponding to params
        
        Update rules:
            t = t + 1
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * g^2
            m_hat = m / (1 - beta1^t)
            v_hat = v / (1 - beta2^t)
            param = param - lr * m_hat / (sqrt(v_hat) + eps)
        """
        # TODO: Implement this
        pass

# Test your implementation
def test_simple_adam():
    np.random.seed(42)
    W = np.random.randn(10, 5)
    
    optimizer = SimpleAdam([W], lr=0.01)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    losses = []
    for epoch in range(100):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        grad = (2 / len(X)) * X.T @ (y_pred - y)
        
        result = optimizer.step([grad])
        if result is None and optimizer.m is None:
            print("Not implemented yet!")
            return
    
    print(f"Initial loss: {losses[0]:.4f}")
    print(f"Final loss: {losses[-1]:.4f}")

test_simple_adam()

<details>
<summary><b>Click for Solution</b></summary>

```python
class SimpleAdam:
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        
        # Initialize moments
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0
    
    def step(self, grads):
        self.t += 1
        
        for i, (param, grad) in enumerate(zip(self.params, grads)):
            # Update biased first moment
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            
            # Update biased second moment
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * grad**2
            
            # Bias correction
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            
            # Update parameters
            param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)
```
</details>

In [ ]:
# SOLUTION

class SimpleAdamSolution:
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        
        # Initialize moments
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0
    
    def step(self, grads):
        self.t += 1
        
        for i, (param, grad) in enumerate(zip(self.params, grads)):
            # Update biased first moment
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            
            # Update biased second moment
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * grad**2
            
            # Bias correction
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            
            # Update parameters
            param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# Test solution
def test_solution_adam():
    np.random.seed(42)
    W = np.random.randn(10, 5)
    
    optimizer = SimpleAdamSolution([W], lr=0.01)
    
    X = np.random.randn(100, 10)
    y = np.random.randn(100, 5)
    
    losses = []
    for epoch in range(100):
        y_pred = X @ W
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        
        grad = (2 / len(X)) * X.T @ (y_pred - y)
        optimizer.step([grad])
    
    print(f"Solution - Initial loss: {losses[0]:.4f}")
    print(f"Solution - Final loss: {losses[-1]:.4f}")
    print(f"Solution - Loss decreased: {losses[-1] < losses[0]}")

test_solution_adam()

---

## Summary

**Key Topics Covered:**

1. Optimizer comparisons (Adam vs SGD, RMSprop)
2. Gradient problems (vanishing, exploding)
3. Learning rate selection and scheduling
4. Momentum and adaptive learning rates
5. Weight decay vs L2 regularization
6. Batch normalization
7. Adam components and bias correction
8. Second-order methods
9. Debugging training issues

**Coding Skills:**
- Implementing SGD with momentum
- Learning rate finder
- Adam from scratch

---
*Generated for "Optimization for Machine Learning" book*